# Create catalog databases

Creates two **Glue databases** (`pf_silver`, `pf_gold`) and the **Athena workgroup** `pf-wg`.
Run the cells top to bottom. Each cell is safe to run twice, and prints the AWS CLI command it runs (`$ aws ...`).
Names and region come from `config/settings.json`.

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# aws() runs the AWS CLI and prints the command and its output (see infra/aws_helpers.py)
from aws_helpers import aws, aws_json, whoami, SETTINGS, RESULTS

DATABASES = SETTINGS["glue_databases"]        # {"silver": "pf_silver", "gold": "pf_gold"}
WORKGROUP = SETTINGS["athena_workgroup"]      # "pf-wg"
SCAN_LIMIT_BYTES = 1024 ** 3                  # 1 GB: Athena cancels any query that scans more

print("identity :", whoami())
print("databases:", list(DATABASES.values()))
print("workgroup:", WORKGROUP)
print("results  :", f"s3://{RESULTS}/")

identity : user/Ahmed
databases: ['pf_silver', 'pf_gold']
workgroup: pf-wg
results  : s3://patientflow-athena-query-results-zhmkbf/


## 2. Check the results bucket exists
The workgroup writes its query results there, so it must exist first (notebook `s3_buckets`).

In [2]:
found = aws("s3api", "head-bucket", "--bucket", RESULTS, check=False).returncode == 0
assert found, f"bucket {RESULTS} does not exist - run the s3_buckets notebook first"
print("OK  results bucket exists")

$ aws s3api head-bucket --bucket patientflow-athena-query-results-zhmkbf


{
    "BucketArn": "arn:aws:s3:::patientflow-athena-query-results-zhmkbf",
    "BucketRegion": "us-east-1",
    "AccessPointAlias": false
}
OK  results bucket exists


## 3. Glue databases
A Glue database is a named folder in the Glue Data Catalog. It holds table definitions, not data.
Athena and Glue jobs find tables through it.

In [3]:
DESCRIPTIONS = {
    "silver": "Cleaned and flattened FHIR data (Parquet)",
    "gold": "Dimensional model and KPI views (Iceberg)",
}

for layer, name in DATABASES.items():
    exists = aws("glue", "get-database", "--name", name, check=False, show=False).returncode == 0
    if exists:
        print(f"database {name} already exists - skipping\n")
    else:
        aws("glue", "create-database",
            "--database-input", json.dumps({"Name": name, "Description": DESCRIPTIONS[layer]}))

$ aws glue get-database --name pf_silver


database pf_silver already exists - skipping

$ aws glue get-database --name pf_gold


database pf_gold already exists - skipping



## 4. Athena workgroup
A workgroup holds query settings. Ours:

| Setting | Value | Effect |
|---|---|---|
| Results location | `s3://<results bucket>/` | where every query result is saved |
| Enforce workgroup settings | true | a query cannot override the location |
| Scan limit per query | 1 GB | a query that would scan more is cancelled |
| Engine | Athena engine version 3 | supports Iceberg `MERGE` |

In [4]:
RESULT_CONFIG = {
    "OutputLocation": f"s3://{RESULTS}/",
    "EncryptionConfiguration": {"EncryptionOption": "SSE_S3"},
}
ENGINE = {"SelectedEngineVersion": "Athena engine version 3"}

exists = aws("athena", "get-work-group", "--work-group", WORKGROUP, check=False, show=False).returncode == 0

if exists:
    # Same settings again: the update call keeps an existing workgroup in line with this notebook
    print(f"workgroup {WORKGROUP} already exists - updating its settings\n")
    aws("athena", "update-work-group", "--work-group", WORKGROUP, "--configuration-updates", json.dumps({
        "EnforceWorkGroupConfiguration": True,
        "ResultConfigurationUpdates": RESULT_CONFIG,
        "BytesScannedCutoffPerQuery": SCAN_LIMIT_BYTES,
        "EngineVersion": ENGINE,
    }))
else:
    aws("athena", "create-work-group", "--name", WORKGROUP,
        "--description", "Patient-flow pipeline queries",
        "--configuration", json.dumps({
            "EnforceWorkGroupConfiguration": True,
            "ResultConfiguration": RESULT_CONFIG,
            "BytesScannedCutoffPerQuery": SCAN_LIMIT_BYTES,
            "EngineVersion": ENGINE,
        }))

$ aws athena get-work-group --work-group pf-wg


workgroup pf-wg already exists - updating its settings

$ aws athena update-work-group --work-group pf-wg --configuration-updates {"EnforceWorkGroupConfiguration": true, "ResultConfigurationUpdates": {"OutputLocation": "s3://patientflow-athena-query-results-zhmkbf/", "EncryptionConfiguration": {"EncryptionOption": "SSE_S3"}}, "BytesScannedCutoffPerQuery": 1073741824, "EngineVersion": {"SelectedEngineVersion": "Athena engine version 3"}}


## 5. Verify
Read everything back from AWS; an `assert` fails if anything differs from the design.

In [5]:
listed = aws_json("glue", "get-databases")["DatabaseList"]
names = {d["Name"] for d in listed}
for name in DATABASES.values():
    assert name in names, f"Glue database {name} is missing"
    print(f"OK  Glue database {name}")

wg = aws_json("athena", "get-work-group", "--work-group", WORKGROUP)["WorkGroup"]
cfg = wg["Configuration"]
assert wg["State"] == "ENABLED"
assert cfg["EnforceWorkGroupConfiguration"] is True
assert cfg["BytesScannedCutoffPerQuery"] == SCAN_LIMIT_BYTES
assert cfg["ResultConfiguration"]["OutputLocation"] == f"s3://{RESULTS}/"
assert cfg["EngineVersion"]["EffectiveEngineVersion"].startswith("Athena engine version 3")
print(f"OK  workgroup {WORKGROUP}: enforced, scan limit {cfg['BytesScannedCutoffPerQuery'] / 1024**3:.0f} GB, "
      f"{cfg['EngineVersion']['EffectiveEngineVersion']}")

OK  Glue database pf_silver
OK  Glue database pf_gold


OK  workgroup pf-wg: enforced, scan limit 1 GB, Athena engine version 3


## What these are for

| Resource | Used by |
|---|---|
| `pf_silver` | the Glue job registers the silver tables here (Phase 5) |
| `pf_gold` | the gold Iceberg tables and KPI views are created here (Phase 6) |
| `pf-wg` | every Athena query: the gold `MERGE`s, the data-quality checks and the notebooks |

The Lake Formation permission mode is **not** switched on; access stays IAM-based.

**Next:** `iam_roles` creates the four roles the services will use.